# Baseline Models — XGBoost & LightGBM
> **Research context:** These gradient-boosted tree models serve as the benchmark.
> The deep-learning models (TabTransformer, GNN) need to beat these numbers to
> justify their added complexity.

**Primary metric: PR-AUC** (precision-recall area under curve).
With only ~3.5% fraud, accuracy is meaningless — a model that never predicts fraud
scores 96.5%. PR-AUC measures how well the model separates fraud from non-fraud
across all operating thresholds.

In [ ]:
# Install required packages (run once in Colab)
!pip install -q xgboost lightgbm pyarrow pandas numpy scikit-learn

## 1 · Data Configuration
Set `DATA_DIR` to wherever your raw CSVs live.

In [ ]:
import os
from google.colab import drive

# ── Mount Google Drive and point to your data ─────────────────────────────────
# Before running:
#   1. Upload train_transaction.csv and train_identity.csv to Google Drive
#      e.g. into a folder called "ieee_fraud/raw/"
#   2. Update DRIVE_DATA_PATH below to match where you put them
#   3. Run this cell — it will prompt you to authorise Drive access

drive.mount("/content/drive")

DRIVE_DATA_PATH = "/content/drive/MyDrive/ieee_fraud/raw"  # ← update this if needed
DATA_DIR = "data"

os.makedirs(f"{DATA_DIR}/raw",     exist_ok=True)
os.makedirs(f"{DATA_DIR}/interim", exist_ok=True)

# Copy from Drive to Colab local storage (faster reads during training)
os.system(f"cp {DRIVE_DATA_PATH}/train_transaction.csv {DATA_DIR}/raw/")
os.system(f"cp {DRIVE_DATA_PATH}/train_identity.csv    {DATA_DIR}/raw/")

print("Files ready:")
os.system(f"ls -lh {DATA_DIR}/raw/")

## 2 · Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 3 · Load & Merge Data
The dataset has two source files that must be joined:
- `train_transaction.csv` — 590k rows, 394 columns (amounts, card info, V-features)
- `train_identity.csv` — 144k rows, 41 columns (device, browser, network info)

~76% of transactions have no identity record (NaN after left join).

In [ ]:
import os
import pandas as pd

def load_merged_train(data_dir="data"):
    """
    Load and merge the IEEE-CIS training files.

    Reads train_transaction.csv and train_identity.csv from data_dir/raw/,
    left-joins them on TransactionID so every transaction is kept (identity
    features are NaN for ~76% of rows that have no identity record), then
    caches the result to data_dir/interim/merged_train.parquet for fast
    re-loads.

    Returns
    -------
    pd.DataFrame  shape ~ (590540, 434)
    """
    cache = os.path.join(data_dir, "interim", "merged_train.parquet")
    if os.path.exists(cache):
        print(f"Loading from cache: {cache}")
        df = pd.read_parquet(cache)
        print(f"Shape: {df.shape}")
        return df

    raw = os.path.join(data_dir, "raw")
    print("Reading train_transaction.csv …")
    trans = pd.read_csv(os.path.join(raw, "train_transaction.csv"))
    print("Reading train_identity.csv …")
    ident = pd.read_csv(os.path.join(raw, "train_identity.csv"))
    print("Merging on TransactionID …")
    df = trans.merge(ident, how="left", on="TransactionID")
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    df.to_parquet(cache, index=False)
    print(f"Cached to {cache}.  Shape: {df.shape}")
    return df

df = load_merged_train(DATA_DIR)
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(f"Sorted by TransactionDT.  Shape: {df.shape}")

## 4 · Feature Selection
Automatically picks numeric and categorical columns using heuristic rules.
Drops columns that are >95% missing and caps the total feature count.

In [ ]:
import numpy as np

def auto_select_features(df, target_col="isFraud", max_numeric=50, max_categorical=20):
    """
    Heuristic feature selection for the IEEE-CIS dataset.

    Rules applied in order:
      1. Drop columns that are >95% missing (too sparse to be useful).
      2. Classify remaining columns as numeric or categorical:
           - object / category dtype  → categorical
           - integer with ≤20 unique values → categorical (likely encoded flag)
           - integer with >20 unique values → numeric
           - float → numeric
      3. Cap at max_numeric + max_categorical to keep model sizes manageable.

    Returns
    -------
    numeric_cols : list[str]
    categorical_cols : list[str]
    """
    ignore = {target_col, "TransactionID"}
    na_frac = df.isna().mean()
    kept = [c for c in na_frac[na_frac < 0.95].index if c not in ignore]

    numeric_cols, categorical_cols = [], []
    for col in kept:
        dt = df[col].dtype
        if dt == "object" or str(dt) == "category":
            categorical_cols.append(col)
        elif np.issubdtype(dt, np.integer):
            (categorical_cols if df[col].nunique(dropna=True) <= 20
             else numeric_cols).append(col)
        elif np.issubdtype(dt, np.floating):
            numeric_cols.append(col)

    numeric_cols     = numeric_cols[:max_numeric]
    categorical_cols = categorical_cols[:max_categorical]
    print(f"Selected {len(numeric_cols)} numeric, {len(categorical_cols)} categorical features")
    print("Numeric:", numeric_cols)
    print("Categorical:", categorical_cols)
    return numeric_cols, categorical_cols

numeric_cols, categorical_cols = auto_select_features(
    df, target_col="isFraud", max_numeric=50, max_categorical=20
)

## 5 · Temporal Train / Val / Test Split
Transactions are sorted by `TransactionDT` (seconds since reference date) before
splitting, so the model is always trained on older data and evaluated on newer data.
Split: **70% train · 15% val · 15% test**.

In [ ]:
# Temporal 70 / 15 / 15 split — respects transaction time ordering.
# Using random splits on time-series data leaks future info into training.
n        = len(df)
n_train  = int(0.70 * n)
n_val    = int(0.15 * n)
train_idx = list(range(0, n_train))
val_idx   = list(range(n_train, n_train + n_val))
test_idx  = list(range(n_train + n_val, n))
print(f"Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}")

## 6 · Feature Preparation for Tree Models
Tree models do not need scaling.  We only need to:
- Label-encode categorical columns (`.cat.codes` gives -1 for NaN, which XGBoost handles)
- Fill numeric NaNs with 0

In [ ]:
def prepare_features(df, numeric_cols, categorical_cols, target_col="isFraud"):
    """
    Minimal preprocessing for gradient-boosted trees.
    Categoricals are label-encoded; numerics are NaN-filled with 0.
    Trees handle both without scaling.
    """
    X = df[numeric_cols + categorical_cols].copy()
    y = df[target_col].values
    for col in categorical_cols:
        X[col] = X[col].astype("category").cat.codes  # -1 for NaN
    X[numeric_cols] = X[numeric_cols].fillna(0)
    return X.values.astype(np.float32), y

X, y = prepare_features(df, numeric_cols, categorical_cols)
X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos
print(f"Positive (fraud) samples in train: {int(pos):,}")
print(f"scale_pos_weight = {scale_pos_weight:.2f}  (penalises missing fraud)")

## 7 · Evaluation Helper
`tune_threshold_and_eval` sweeps the precision-recall curve to find the
decision threshold that maximises F1, then reports all metrics at that threshold.
This gives a fairer comparison than a fixed 0.5 threshold on imbalanced data.

In [ ]:
def tune_threshold_and_eval(y_true, y_score, label):
    """
    Find the threshold maximising F1 on the PR curve,
    then return precision, recall, F1, ROC-AUC, and PR-AUC.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_thr = float(thresholds[f1s.argmax()])
    y_pred   = (y_score >= best_thr).astype(int)
    return {
        f"{label}_precision":      precision_score(y_true, y_pred, zero_division=0),
        f"{label}_recall":         recall_score(y_true, y_pred, zero_division=0),
        f"{label}_f1":             f1_score(y_true, y_pred, zero_division=0),
        f"{label}_roc_auc":        roc_auc_score(y_true, y_score),
        f"{label}_pr_auc":         average_precision_score(y_true, y_score),
        f"{label}_best_threshold": best_thr,
    }

def run_model(name, clf, X_train, y_train, X_val, y_val, X_test, y_test):
    """Train a sklearn-compatible classifier and evaluate on val + test sets."""
    print(f"\nTraining {name} …")
    clf.fit(X_train, y_train)
    val_m  = tune_threshold_and_eval(y_val,  clf.predict_proba(X_val)[:, 1],  "val")
    test_m = tune_threshold_and_eval(y_test, clf.predict_proba(X_test)[:, 1], "test")
    print(f"\n{name} — Validation")
    for k, v in val_m.items():  print(f"  {k:30s}: {v:.4f}")
    print(f"\n{name} — Test")
    for k, v in test_m.items(): print(f"  {k:30s}: {v:.4f}")
    return {**val_m, **test_m}

## 8 · XGBoost
XGBoost uses gradient-boosted decision trees with second-order gradient statistics.
`scale_pos_weight` compensates for class imbalance by upweighting fraud examples.
`eval_metric="aucpr"` guides the internal evaluation toward PR-AUC.

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_metrics = run_model("XGBoost", xgb,
                         X_train, y_train, X_val, y_val, X_test, y_test)

## 9 · LightGBM
LightGBM is a faster alternative to XGBoost using histogram-based splits and
leaf-wise tree growth. Often achieves similar accuracy with lower memory usage.

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgbm_metrics = run_model("LightGBM", lgbm,
                          X_train, y_train, X_val, y_val, X_test, y_test)

## 10 · Results Summary

In [ ]:
print("\n" + "="*45)
print("  SUMMARY — Test Set")
print("="*45)
print(f"  XGBoost   PR-AUC : {xgb_metrics['test_pr_auc']:.4f}")
print(f"  XGBoost   ROC-AUC: {xgb_metrics['test_roc_auc']:.4f}")
print(f"  XGBoost   F1     : {xgb_metrics['test_f1']:.4f}")
print()
print(f"  LightGBM  PR-AUC : {lgbm_metrics['test_pr_auc']:.4f}")
print(f"  LightGBM  ROC-AUC: {lgbm_metrics['test_roc_auc']:.4f}")
print(f"  LightGBM  F1     : {lgbm_metrics['test_f1']:.4f}")